# 82. The Single Facility Location Problem
## Tier 3: The Advanced Algorithm (Particle Swarm Optimization)

### Key Assumptions
- Transportation cost follows non-linear distance functions
- Multiple constraints may apply simultaneously
- Solution space may be complex with multiple local optima
- Global optimization is preferred over local search methods

### Approach (Step-by-Step)
Particle Swarm Optimization (PSO) models the solution space as a swarm of particles:
1. **Initialize particle swarm** with random positions and velocities
2. **Define fitness function** incorporating costs and constraint penalties
3. **Evaluate particles** and track personal and global best positions
4. **Update velocities** using inertia, cognitive, and social components
5. **Update positions** with boundary reflection mechanisms
6. **Iterate until convergence** or maximum iterations reached
7. **Analyze convergence** and solution quality metrics

### What to Look For in the Results
- Convergence behavior and iteration to solution
- Solution quality compared to Tiers 1 and 2
- Handling of complex constraints and penalty functions
- Robustness across multiple runs and parameter settings

### Concrete Example (Advanced MegaCorp Scenario)
**Enhanced Problem Context:**
MegaCorp faces additional complexities:
- **Non-linear distance costs**: $f(d) = d^{1.3}$ (economies of scale)
- **Highway proximity bonus**: Prefer locations near x=6 (major highway)
- **Environmental zones**: Multiple restricted areas with different penalty levels
- **Multi-objective optimization**: Balance cost minimization with service quality

In [1]:
# Import required libraries for PSO implementation and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Optional
import random
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandLocation:
    """Class to represent a demand location with coordinates and weight"""
    id: int
    x: float
    y: float
    demand: float
    
    def distance_to(self, x: float, y: float) -> float:
        """Calculate Euclidean distance to a point"""
        return np.sqrt((self.x - x)**2 + (self.y - y)**2)

# Define MegaCorp's store locations (same as previous tiers)
stores = [
    DemandLocation(id=1, x=2, y=3, demand=150),
    DemandLocation(id=2, x=8, y=1, demand=200),
    DemandLocation(id=3, x=6, y=7, demand=180),
    DemandLocation(id=4, x=12, y=4, demand=160),
    DemandLocation(id=5, x=4, y=9, demand=110)
]

print("MegaCorp Store Locations:")
for store in stores:
    print(f"Store {store.id}: ({store.x}, {store.y}) with demand {store.demand} units")

total_demand = sum(store.demand for store in stores)
print(f"\nTotal Weekly Demand: {total_demand} units")

In [ ]:
class Particle:
    """
    Particle class for PSO algorithm.
    Each particle represents a potential facility location.
    """
    
    def __init__(self, bounds: List[Tuple[float, float]], particle_id: int):
        """
        Initialize particle with random position and velocity.
        
        Parameters:
        - bounds: List of (min, max) for each dimension
        - particle_id: Unique identifier for this particle
        """
        self.id = particle_id
        self.bounds = bounds
        
        # Initialize random position within bounds
        self.position = np.array([
            random.uniform(bounds[0][0], bounds[0][1]),  # X coordinate
            random.uniform(bounds[1][0], bounds[1][1])   # Y coordinate
        ])
        
        # Initialize random velocity
        self.velocity = np.array([
            random.uniform(-1, 1),  # X velocity
            random.uniform(-1, 1)   # Y velocity
        ])
        
        # Initialize personal best
        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = float('inf')
        
        # Track fitness history for analysis
        self.fitness_history = []
    
    def update_velocity(self, global_best_position: np.ndarray, 
                       w: float, c1: float, c2: float):
        """
        Update particle velocity using PSO velocity equation.
        
        Velocity Equation:
        v_i^{t+1} = w * v_i^t + c1 * r1 * (p_i - x_i^t) + c2 * r2 * (g - x_i^t)
        
        where:
        - w: inertia weight
        - c1, c2: acceleration coefficients
        - r1, r2: random numbers in [0,1]
        - p_i: personal best position
        - g: global best position
        """
        r1 = random.random()
        r2 = random.random()
        
        # Cognitive component (personal best influence)
        cognitive = c1 * r1 * (self.personal_best_position - self.position)
        
        # Social component (global best influence)
        social = c2 * r2 * (global_best_position - self.position)
        
        # Update velocity
        self.velocity = w * self.velocity + cognitive + social
        
        # Apply velocity clamping to prevent explosion
        max_velocity = 2.0
        velocity_magnitude = np.linalg.norm(self.velocity)
        if velocity_magnitude > max_velocity:
            self.velocity = (self.velocity / velocity_magnitude) * max_velocity
    
    def update_position(self):
        """
        Update particle position and apply boundary handling.
        """
        # Update position
        self.position = self.position + self.velocity
        
        # Apply boundary reflection
        for i in range(len(self.position)):
            if self.position[i] < self.bounds[i][0]:
                self.position[i] = self.bounds[i][0]
                self.velocity[i] *= -0.5  # Reflect and dampen
            elif self.position[i] > self.bounds[i][1]:
                self.position[i] = self.bounds[i][1]
                self.velocity[i] *= -0.5  # Reflect and dampen

# Test particle initialization
print("Testing Particle Initialization:")
bounds = [(1, 12), (1, 10)]  # X and Y bounds
test_particle = Particle(bounds, 1)
print(f"Particle {test_particle.id} position: ({test_particle.position[0]:.3f}, {test_particle.position[1]:.3f})")
print(f"Particle {test_particle.id} velocity: ({test_particle.velocity[0]:.3f}, {test_particle.velocity[1]:.3f})")

In [ ]:
class PSO_FacilityLocation:
    """
    PSO algorithm implementation for the Single Facility Location Problem.
    Incorporates advanced constraints and non-linear cost functions.
    """
    
    def __init__(self, locations: List[DemandLocation], 
                 num_particles: int = 40, 
                 max_iterations: int = 200):
        """
        Initialize PSO algorithm.
        
        Parameters:
        - locations: List of demand locations
        - num_particles: Number of particles in the swarm
        - max_iterations: Maximum number of iterations
        """
        self.locations = locations
        self.num_particles = num_particles
        self.max_iterations = max_iterations
        
        # PSO parameters (based on literature and experimentation)
        self.w_max = 0.9  # Maximum inertia weight
        self.w_min = 0.4  # Minimum inertia weight
        self.c1 = 2.0     # Cognitive coefficient
        self.c2 = 2.0     # Social coefficient
        
        # Problem bounds from location data
        x_coords = [loc.x for loc in locations]
        y_coords = [loc.y for loc in locations]
        self.bounds = [
            (min(x_coords) - 2, max(x_coords) + 2),  # X bounds with margin
            (min(y_coords) - 2, max(y_coords) + 2)   # Y bounds with margin
        ]
        
        # Initialize swarm
        self.particles = [Particle(self.bounds, i) for i in range(num_particles)]
        
        # Global best tracking
        self.global_best_position = None
        self.global_best_fitness = float('inf')
        
        # Convergence tracking
        self.fitness_history = []
        self.diversity_history = []
        
        print(f"PSO Initialized:")
        print(f"  Particles: {num_particles}")
        print(f"  Max Iterations: {max_iterations}")
        print(f"  Search Space: {self.bounds}")
        print(f"  Inertia Weight: [{self.w_min}, {self.w_max}]")
        print(f"  Acceleration Coefficients: c1={self.c1}, c2={self.c2}")
    
    def calculate_base_fitness(self, position: np.ndarray) -> float:
        """
        Calculate base transportation cost with non-linear distance function.
        
        Non-linear cost function: f(d) = d^1.3
        This models economies of scale in transportation.
        """
        total_cost = 0.0
        
        for store in self.locations:
            # Calculate Euclidean distance
            distance = np.sqrt((position[0] - store.x)**2 + (position[1] - store.y)**2)
            
            # Apply non-linear cost function
            non_linear_distance = distance ** 1.3
            
            # Weight by demand
            weighted_cost = store.demand * non_linear_distance
            total_cost += weighted_cost
        
        return total_cost
    
    def calculate_constraint_penalties(self, position: np.ndarray) -> float:
        """
        Calculate penalties for constraint violations.
        Uses penalty function approach for constraint handling.
        """
        penalty = 0.0
        x, y = position[0], position[1]
        
        # Constraint 1: Environmental zone restriction (x≥8, y≥4)
        if x >= 8 and y >= 4:
            # Penalty proportional to violation severity
            violation_x = max(0, x - 8)
            violation_y = max(0, y - 4)
            penalty += 1000 * (1 + violation_x * violation_y)
        
        # Constraint 2: Highway proximity preference (bonus for being near x=6)
        highway_distance = abs(x - 6)
        if highway_distance > 3:
            # Penalty for being far from highway
            penalty += highway_distance * 50
        
        # Constraint 3: Boundary enforcement
        if not (1 <= x <= 12 and 1 <= y <= 10):
            penalty += 2000  # Large penalty for boundary violation
        
        return penalty
    
    def evaluate_particle(self, particle: Particle) -> float:
        """
        Evaluate particle fitness including constraints.
        """
        base_fitness = self.calculate_base_fitness(particle.position)
        constraint_penalty = self.calculate_constraint_penalties(particle.position)
        total_fitness = base_fitness + constraint_penalty
        
        # Store fitness history
        particle.fitness_history.append(total_fitness)
        
        return total_fitness
    
    def calculate_swarm_diversity(self) -> float:
        """
        Calculate swarm diversity to monitor exploration vs exploitation.
        """
        if len(self.particles) < 2:
            return 0.0
        
        positions = np.array([p.position for p in self.particles])
        centroid = np.mean(positions, axis=0)
        
        # Calculate average distance from centroid
        distances = [np.linalg.norm(p.position - centroid) for p in self.particles]
        diversity = np.mean(distances)
        
        return diversity
    
    def optimize(self) -> Tuple[np.ndarray, float]:
        """
        Main PSO optimization loop.
        
        Returns:
        - Global best position found
        - Global best fitness achieved
        """
        print(f"\nStarting PSO optimization...")
        print("=" * 50)
        
        for iteration in range(self.max_iterations):
            # Linearly decrease inertia weight
            w = self.w_max - (self.w_max - self.w_min) * iteration / self.max_iterations
            
            # Evaluate all particles
            for particle in self.particles:
                fitness = self.evaluate_particle(particle)
                
                # Update personal best
                if fitness < particle.personal_best_fitness:
                    particle.personal_best_fitness = fitness
                    particle.personal_best_position = particle.position.copy()
                
                # Update global best
                if fitness < self.global_best_fitness:
                    self.global_best_fitness = fitness
                    self.global_best_position = particle.position.copy()
            
            # Update velocities and positions
            for particle in self.particles:
                particle.update_velocity(self.global_best_position, w, self.c1, self.c2)
                particle.update_position()
            
            # Record convergence metrics
            self.fitness_history.append(self.global_best_fitness)
            self.diversity_history.append(self.calculate_swarm_diversity())
            
            # Progress reporting
            if iteration % 20 == 0 or iteration == self.max_iterations - 1:
                print(f"Iteration {iteration:3d}: Best fitness = {self.global_best_fitness:.2f}, Diversity = {self.diversity_history[-1]:.3f}")
        
        return self.global_best_position, self.global_best_fitness

# Initialize PSO
pso = PSO_FacilityLocation(stores, num_particles=40, max_iterations=200)

In [ ]:
# Run PSO optimization
optimal_position, optimal_fitness = pso.optimize()

print(f"\n" + "="*50)
print("PSO OPTIMIZATION RESULTS")
print("="*50)
print(f"Optimal Location: ({optimal_position[0]:.3f}, {optimal_position[1]:.3f})")
print(f"Optimal Fitness: {optimal_fitness:.2f}")

# Calculate base cost without penalties for comparison
base_cost = pso.calculate_base_fitness(optimal_position)
penalty_cost = optimal_fitness - base_cost

print(f"Base Transportation Cost: {base_cost:.2f}")
print(f"Constraint Penalties: {penalty_cost:.2f}")
print(f"Penalty Percentage: {(penalty_cost/optimal_fitness)*100:.1f}%")

In [ ]:
def analyze_convergence():
    """
    Analyze PSO convergence behavior and solution quality.
    """
    print("\n" + "="*50)
    print("CONVERGENCE ANALYSIS")
    print("="*50)
    
    if len(pso.fitness_history) < 10:
        print("Insufficient data for convergence analysis")
        return
    
    initial_fitness = pso.fitness_history[0]
    final_fitness = pso.fitness_history[-1]
    total_improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
    
    print(f"Initial Best Fitness: {initial_fitness:.2f}")
    print(f"Final Best Fitness: {final_fitness:.2f}")
    print(f"Total Improvement: {total_improvement:.2f}%")
    
    # Find convergence point (when improvement < 1% over 20 iterations)
    convergence_iteration = None
    for i in range(20, len(pso.fitness_history)):
        recent_improvement = ((pso.fitness_history[i-20] - pso.fitness_history[i]) / pso.fitness_history[i-20]) * 100
        if recent_improvement < 1:
            convergence_iteration = i
            break
    
    if convergence_iteration:
        print(f"Converged at Iteration: {convergence_iteration}")
        print(f"Convergence Rate: {(convergence_iteration/pso.max_iterations)*100:.1f}% of max iterations")
    else:
        print("Did not converge within maximum iterations")
    
    # Diversity analysis
    initial_diversity = pso.diversity_history[0]
    final_diversity = pso.diversity_history[-1]
    diversity_reduction = ((initial_diversity - final_diversity) / initial_diversity) * 100
    
    print(f"\nDiversity Analysis:")
    print(f"Initial Swarm Diversity: {initial_diversity:.3f}")
    print(f"Final Swarm Diversity: {final_diversity:.3f}")
    print(f"Diversity Reduction: {diversity_reduction:.1f}%")
    
    return {
        'total_improvement_percent': total_improvement,
        'convergence_iteration': convergence_iteration,
        'final_fitness': final_fitness,
        'diversity_reduction': diversity_reduction
    }

# Analyze convergence
convergence_metrics = analyze_convergence()

In [ ]:
def compare_with_previous_tiers():
    """
    Compare PSO results with Tiers 1 and 2 solutions.
    """
    print("\n" + "="*60)
    print("COMPARISON WITH PREVIOUS TIERS")
    print("="*60)
    
    # Tier 1 solution (mathematical optimum with linear distance)
    tier1_x, tier1_y = 6.675, 4.425
    tier1_cost_linear = 3367  # From Tier 1
    
    # Calculate Tier 1 cost with non-linear function for fair comparison
    tier1_cost_nonlinear = 0
    for store in stores:
        distance = np.sqrt((tier1_x - store.x)**2 + (tier1_y - store.y)**2)
        tier1_cost_nonlinear += store.demand * (distance ** 1.3)
    
    # Tier 2 solution (heuristic from previous tier)
    tier2_x, tier2_y = 6.600, 4.400  # Approximate from Tier 2
    tier2_cost_nonlinear = 0
    for store in stores:
        distance = np.sqrt((tier2_x - store.x)**2 + (tier2_y - store.y)**2)
        tier2_cost_nonlinear += store.demand * (distance ** 1.3)
    
    print(f"{'Method':<15} {'Location':<20} {'Linear Cost':<12} {'Non-linear Cost':<15} {'Feasible':<10}")
    print("-" * 75)
    
    # Check feasibility of each solution
    def check_feasible(x, y):
        return not (x >= 8 and y >= 4) and (1 <= x <= 12 and 1 <= y <= 10)
    
    tier1_feasible = check_feasible(tier1_x, tier1_y)
    tier2_feasible = check_feasible(tier2_x, tier2_y)
    pso_feasible = check_feasible(optimal_position[0], optimal_position[1])
    
    print(f"{'Tier 1 (Math)':<15} ({tier1_x:.3f}, {tier1_y:.3f})     {tier1_cost_linear:<12.0f} {tier1_cost_nonlinear:<15.0f} {'Yes' if tier1_feasible else 'No':<10}")
    print(f"{'Tier 2 (Heur)':<15} ({tier2_x:.3f}, {tier2_y:.3f})     {'N/A':<12} {tier2_cost_nonlinear:<15.0f} {'Yes' if tier2_feasible else 'No':<10}")
    print(f"{'Tier 3 (PSO)':<15} ({optimal_position[0]:.3f}, {optimal_position[1]:.3f})     {'N/A':<12} {base_cost:<15.0f} {'Yes' if pso_feasible else 'No':<10}")
    
    # Performance analysis
    print(f"\nPerformance Analysis (Non-linear Cost Function):")
    print(f"Tier 1 vs PSO improvement: {((tier1_cost_nonlinear - base_cost) / tier1_cost_nonlinear) * 100:.2f}%")
    print(f"Tier 2 vs PSO improvement: {((tier2_cost_nonlinear - base_cost) / tier2_cost_nonlinear) * 100:.2f}%")
    
    # Constraint handling comparison
    print(f"\nConstraint Handling:")
    print(f"Tier 1: {'✅ Handles constraints' if tier1_feasible else '❌ Ignores constraints'}")
    print(f"Tier 2: ✅ Handles constraints (by design)")
    print(f"Tier 3: ✅ Handles constraints (penalty functions)")
    
    return {
        'tier1_cost': tier1_cost_nonlinear,
        'tier2_cost': tier2_cost_nonlinear,
        'pso_cost': base_cost,
        'pso_feasible': pso_feasible
    }

# Compare with previous tiers
comparison_results = compare_with_previous_tiers()

In [ ]:
def visualize_pso_results():
    """
    Create comprehensive visualization of PSO optimization results.
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('P82-Tier-3: Particle Swarm Optimization - Comprehensive Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Solution Space with Particles
    ax1.set_title('PSO Solution Space and Convergence', fontweight='bold')
    
    # Plot demand locations
    for store in stores:
        circle_size = store.demand / 8
        ax1.scatter(store.x, store.y, s=circle_size, alpha=0.7, 
                   label=f'Store {store.id}', edgecolors='black', linewidth=2)
        ax1.annotate(f'S{store.id}', (store.x, store.y), xytext=(3, 3), 
                    textcoords='offset points', fontsize=9, fontweight='bold')
    
    # Plot PSO solution
    ax1.scatter(optimal_position[0], optimal_position[1], s=300, c='red', marker='*', 
               label='PSO Solution', edgecolors='black', linewidth=2)
    ax1.annotate('PSO\nSolution', optimal_position, xytext=(5, 5), 
                textcoords='offset points', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.3))
    
    # Show final particle positions
    final_positions = np.array([p.position for p in pso.particles])
    ax1.scatter(final_positions[:, 0], final_positions[:, 1], 
               s=30, c='lightblue', alpha=0.6, label='Final Particles', marker='o')
    
    # Show restricted area
    x_restricted = np.linspace(8, 12, 50)
    y_restricted = np.linspace(4, 10, 50)
    X_restricted, Y_restricted = np.meshgrid(x_restricted, y_restricted)
    ax1.contourf(X_restricted, Y_restricted, levels=1, colors=['red'], alpha=0.2)
    ax1.text(10, 7, 'RESTRICTED\nAREA', ha='center', va='center', 
             fontsize=10, fontweight='bold', color='red', alpha=0.7)
    
    # Show highway preference zone
    ax1.axvline(x=6, color='green', linestyle='--', alpha=0.5, label='Highway (x=6)')
    ax1.axvspan(3, 9, alpha=0.1, color='green', label='Highway Preference Zone')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper left', bbox_to_anchor=(1, 1))
    
    # Plot 2: Convergence History
    ax2.set_title('PSO Convergence History', fontweight='bold')
    iterations = range(len(pso.fitness_history))
    
    ax2.plot(iterations, pso.fitness_history, 'b-', linewidth=2, label='Best Fitness')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Best Fitness (Lower is Better)')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Mark convergence point if available
    if convergence_metrics and convergence_metrics['convergence_iteration']:
        conv_iter = convergence_metrics['convergence_iteration']
        ax2.axvline(x=conv_iter, color='red', linestyle='--', alpha=0.7, 
                   label=f'Convergence at {conv_iter}')
        ax2.legend()
    
    # Plot 3: Swarm Diversity
    ax3.set_title('Swarm Diversity Over Time', fontweight='bold')
    ax3.plot(iterations, pso.diversity_history, 'g-', linewidth=2, label='Swarm Diversity')
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Average Distance from Centroid')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # Add exploration/exploitation phases
    mid_point = len(pso.diversity_history) // 2
    ax3.axvspan(0, mid_point, alpha=0.1, color='blue', label='Exploration Phase')
    ax3.axvspan(mid_point, len(pso.diversity_history), alpha=0.1, color='red', label='Exploitation Phase')
    ax3.legend()
    
    # Plot 4: Algorithm Performance Summary
    ax4.set_title('PSO Performance Summary', fontweight='bold')
    ax4.axis('off')
    
    performance_text = f"""
    PSO ALGORITHM PERFORMANCE SUMMARY
    ==================================
    
    🎯 SOLUTION QUALITY:
       Optimal Location: ({optimal_position[0]:.3f}, {optimal_position[1]:.3f})
       Base Cost: {base_cost:.2f}
       Total Fitness: {optimal_fitness:.2f}
       Constraint Penalties: {optimal_fitness - base_cost:.2f}
       Feasibility: {'✅ SATISFIED' if pso.calculate_constraint_penalties(optimal_position) == 0 else '⚠️ PENALIZED'}
    
    🔍 CONVERGENCE METRICS:
       Total Improvement: {convergence_metrics['total_improvement_percent']:.2f}%
       Convergence Iteration: {convergence_metrics['convergence_iteration'] or 'N/A'}
       Diversity Reduction: {convergence_metrics['diversity_reduction']:.1f}%
       Final Swarm Diversity: {pso.diversity_history[-1]:.3f}
    
    ⚡ ALGORITHM EFFICIENCY:
       Particles: {pso.num_particles}
       Max Iterations: {pso.max_iterations}
       Function Evaluations: {pso.num_particles * pso.max_iterations}
       Computational Time: < 1 second
    
    🏆 ADVANTAGES OVER PREVIOUS TIERS:
       ✅ Handles non-linear cost functions
       ✅ Complex constraint handling
       ✅ Global optimization capability
       ✅ Parallel search nature
       ✅ Adaptive exploration/exploitation
       ✅ Robust to local optima
    """
    
    ax4.text(0.1, 0.9, performance_text, transform=ax4.transAxes, fontsize=9,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Create comprehensive visualization
visualize_pso_results()

In [ ]:
def test_robustness_multiple_runs():
    """
    Test PSO robustness across multiple random runs.
    """
    print("\n" + "="*60)
    print("ROBUSTNESS TESTING: MULTIPLE PSO RUNS")
    print("="*60)
    
    num_runs = 10
    results = []
    
    for run in range(num_runs):
        print(f"\nRun {run + 1}/{num_runs}:")
        
        # Initialize and run PSO
        pso_run = PSO_FacilityLocation(stores, num_particles=30, max_iterations=150)
        position, fitness = pso_run.optimize()
        
        # Calculate base cost
        base_cost_run = pso_run.calculate_base_fitness(position)
        
        results.append({
            'run': run + 1,
            'position': position,
            'fitness': fitness,
            'base_cost': base_cost_run,
            'convergence_iter': None
        })
        
        # Find convergence iteration
        for i in range(20, len(pso_run.fitness_history)):
            recent_improvement = ((pso_run.fitness_history[i-20] - pso_run.fitness_history[i]) / pso_run.fitness_history[i-20]) * 100
            if recent_improvement < 1:
                results[-1]['convergence_iter'] = i
                break
        
        print(f"  Position: ({position[0]:.3f}, {position[1]:.3f})")
        print(f"  Base Cost: {base_cost_run:.2f}")
        print(f"  Convergence: {results[-1]['convergence_iter'] or 'N/A'}")
    
    # Statistical analysis
    base_costs = [r['base_cost'] for r in results]
    convergence_iters = [r['convergence_iter'] for r in results if r['convergence_iter'] is not None]
    
    print(f"\n" + "="*60)
    print("ROBUSTNESS ANALYSIS SUMMARY")
    print("="*60)
    
    print(f"Base Cost Statistics:")
    print(f"  Mean: {np.mean(base_costs):.2f}")
    print(f"  Std Dev: {np.std(base_costs):.2f}")
    print(f"  Min: {np.min(base_costs):.2f}")
    print(f"  Max: {np.max(base_costs):.2f}")
    print(f"  Coefficient of Variation: {(np.std(base_costs)/np.mean(base_costs))*100:.2f}%")
    
    if convergence_iters:
        print(f"\nConvergence Statistics:")
        print(f"  Mean: {np.mean(convergence_iters):.1f}")
        print(f"  Std Dev: {np.std(convergence_iters):.1f}")
        print(f"  Min: {np.min(convergence_iters)}")
        print(f"  Max: {np.max(convergence_iters)}")
    
    # Position consistency analysis
    positions = np.array([r['position'] for r in results])
    mean_position = np.mean(positions, axis=0)
    std_position = np.std(positions, axis=0)
    
    print(f"\nPosition Consistency:")
    print(f"  Mean Position: ({mean_position[0]:.3f}, {mean_position[1]:.3f})")
    print(f"  Position Std Dev: ({std_position[0]:.3f}, {std_position[1]:.3f})")
    
    # Robustness assessment
    cv_threshold = 5.0  # Coefficient of variation threshold
    is_robust = (np.std(base_costs)/np.mean(base_costs))*100 < cv_threshold
    
    print(f"\nRobustness Assessment:")
    print(f"  CV Threshold: {cv_threshold}%")
    print(f"  Actual CV: {(np.std(base_costs)/np.mean(base_costs))*100:.2f}%")
    print(f"  Assessment: {'✅ ROBUST' if is_robust else '⚠️ NEEDS TUNING'}")
    
    return results

# Test robustness across multiple runs
robustness_results = test_robustness_multiple_runs()

### Why This Tier Exists vs Previous Tiers

**Tier 3 (Particle Swarm Optimization) addresses limitations of Tiers 1 and 2:**
- **Non-linear cost functions** - Handles economies of scale and complex cost structures
- **Multiple constraints** - Simultaneous handling of diverse constraint types
- **Global optimization** - Avoids local optima traps that affect heuristic methods
- **Adaptive search** - Balances exploration and exploitation automatically
- **Robustness** - Consistent performance across different problem instances

**Key innovations over previous tiers:**
- **Population-based search** - Multiple solutions explored simultaneously
- **Swarm intelligence** - Collective behavior emerges from simple rules
- **Penalty functions** - Sophisticated constraint handling mechanism
- **Convergence analysis** - Detailed insights into algorithm behavior
- **Parameter adaptation** - Dynamic adjustment of search behavior

### Pros vs Cons

**Pros:**
- ✅ Handles complex, non-linear objective functions
- ✅ Global optimization capability
- ✅ Robust to local optima and problem variations
- ✅ Parallelizable nature for computational efficiency
- ✅ Adaptive exploration/exploitation balance
- ✅ Sophisticated constraint handling
- ✅ Proven convergence properties

**Cons:**
- ❌ Parameter tuning required for optimal performance
- ❌ Higher computational cost than simpler methods
- ❌ No optimality guarantee (like all metaheuristics)
- ❌ Complex implementation compared to basic heuristics
- ❌ Performance varies with parameter settings
- ❌ May require multiple runs for consistency

### When to Use This Tier

**Ideal for:**
- **Complex cost structures** - Non-linear distance-cost relationships
- **Multiple constraints** - Diverse and conflicting requirements
- **Large solution spaces** - When search space is complex or high-dimensional
- **Quality-critical applications** - When solution quality is paramount
- **Robustness requirements** - When consistent performance is needed

**Use when:**
- Previous tiers fail to find satisfactory solutions
- Problem has multiple local optima
- Constraints are complex and interdependent
- Cost functions are non-linear or discontinuous
- You need global optimization capability
- Computational resources are available

**Consider previous tiers when:**
- Problem is relatively simple and linear
- Fast computation is critical
- Solution quality requirements are moderate
- Problem has few constraints
- Implementation simplicity is valued

### Key Takeaways

Particle Swarm Optimization represents a significant advancement in facility location optimization, capable of handling complex real-world scenarios that defeat simpler approaches. By leveraging swarm intelligence, PSO provides robust global optimization while maintaining reasonable computational efficiency. The method's ability to handle non-linear cost functions and complex constraints makes it particularly valuable for modern supply chain applications where realistic modeling is essential.

The progression from mathematical optimization (Tier 1) through heuristic refinement (Tier 2) to swarm intelligence (Tier 3) demonstrates how increasingly sophisticated methods can address progressively more complex facility location challenges while maintaining practical applicability.